# 05 - Improved logistic purchase probability
Predict `purchase_14d` using the 43 registered pre-send predictors. Purchase probability describes the historical mixture of offers; it is neither treatment assignment propensity nor an untreated response probability.

The model remains logistic regression. We compare raw predictors, signed log transforms, selected behavioral interactions, and spline bases, with regularization. Development uses forward validation on CP04, CP05, and CP06, each trained only on earlier campaigns. The configuration is frozen before CP07 scoring. CP07 was inspected in the earlier audit, so this is retrospective confirmation, **not a fresh untouched holdout**. A future campaign is still needed for prospective confirmation.
The headline purchase metrics are **average precision** and **recall at a development-selected threshold**. Precision, flagged share, and probability-calibration diagnostics accompany recall so a high recall cannot be mistaken for good targeting merely because nearly everyone is flagged.


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.calibration import calibration_curve
from src.common import TABLES, FIGS, MODELS, load_processed
from src.modeling import feature_lists, make_propensity_pipe
from src.propensity import fit_purchase_model, select_purchase_config, purchase_metrics, PURCHASE_SELECTION_RULE
from scripts.experiment_propensity import paired_purchase_intervals, development_purchase_thresholds
pd.set_option("display.width", 200)
df = load_processed("historical_feature_outcome")
NUM, CAT = feature_lists(); X_COLS = NUM + CAT
outcomes = set(pd.read_csv(TABLES / "outcome_schema.csv").outcome)
FORBIDDEN = {"assignment_id", "customer_id", "campaign_id", "send_ts", "outcome_end_ts", "assignment_source", "offer_type", "email_delivered"} | outcomes
assert not set(X_COLS) & FORBIDDEN, set(X_COLS) & FORBIDDEN
development = df[df.campaign_id.lt("CP07")]
test = df[df.campaign_id.eq("CP07")]
print(f"{len(NUM)} numeric + {len(CAT)} categorical = {len(X_COLS)} raw predictors")
df.groupby("campaign_id").purchase_14d.agg(["size", "mean"])

EXPERIMENTS = TABLES.parent / "experiments" / "propensity"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)


36 numeric + 7 categorical = 43 raw predictors


## Development selection and saved configuration
Sixteen configurations compare four feature sets and four regularization strengths. Choose the highest mean **average precision** across forward CP04-CP06 folds, retaining log-loss/Brier safeguards for probability quality and ROC-AUC as the tie-break. The `pr_auc` field remains only an identical compatibility alias for average precision, not trapezoidal PR area. Nested sigmoid calibration is retained only if both probability losses improve. These existing model-selection rules are unchanged; recall is reported at a separately frozen development threshold rather than maximized by predicting every customer will purchase. Development scores are selection-biased.


In [2]:
CONFIG, development_folds, development_summary = select_purchase_config(
    development, NUM, CAT, validation_campaigns=["CP04", "CP05", "CP06"])
development_folds.to_csv(TABLES / "logistic_validation_folds.csv", index=False)
development_summary.to_csv(TABLES / "logistic_validation_grid.csv", index=False)
selection = {"config": CONFIG, "validation_campaigns": ["CP04", "CP05", "CP06"],
             "selection_rule": PURCHASE_SELECTION_RULE,
             "cp07_status": "Previously inspected; retrospective confirmation only"}
(TABLES / "logistic_selected_config.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
print("Frozen configuration:", CONFIG)
print(PURCHASE_SELECTION_RULE)
development_summary.round(6)

development_folds.to_csv(EXPERIMENTS / "development_folds.csv", index=False)
development_summary.to_csv(EXPERIMENTS / "development_summary.csv", index=False)
(EXPERIMENTS / "selection.json").write_text(json.dumps({"config": CONFIG,
    "development_campaigns": ["CP04", "CP05", "CP06"], "rule": PURCHASE_SELECTION_RULE,
    "test_status": "CP07 not used in selection; previously inspected so retrospective confirmation only"}, indent=2), encoding="utf-8")


Frozen configuration: {'mode': 'log_interactions', 'C': 0.01, 'calibration': 'none'}
maximize mean out-of-time average precision with log_loss <= original + .001 and Brier <= original + .0005; ROC-AUC tie-break. Select nested temporal sigmoid only if both log-loss and Brier improve.


479

## Freeze recall thresholds using development predictions only
Each model gets its own threshold from pooled out-of-time CP04-CP06 predictions, using the same maximum-F1 rule and deterministic lowest-threshold tie-break. Thresholds and their provenance are saved **before CP07 predictions or labels are evaluated**. Recall is the fraction of actual purchasers flagged; precision and flagged share show the operating trade-off. This classification threshold is not a promotion allocation rule. The old 0.50 diagnostic remains for compatibility, but is not the headline operating point.


In [3]:
def metrics(y, p, threshold=.5):
    return {**purchase_metrics(y, p, threshold=threshold),
            "base_rate": float(np.mean(y)), "mean_pred": float(np.mean(p))}

threshold_metadata, threshold_predictions = development_purchase_thresholds(development, NUM, CAT, CONFIG)
F1_THR = threshold_metadata["selected"]
ORIGINAL_THR = threshold_metadata["original"]
# Freeze both thresholds before the following cell evaluates CP07.
(TABLES / "purchase_thresholds.json").write_text(json.dumps(threshold_metadata, indent=2)+"\n", encoding="utf-8")
(EXPERIMENTS / "thresholds.json").write_text(json.dumps(threshold_metadata, indent=2)+"\n", encoding="utf-8")
threshold_predictions.to_csv(TABLES / "purchase_threshold_development_predictions.csv", index=False)
threshold_predictions.to_csv(EXPERIMENTS / "threshold_development_predictions.csv", index=False)
validation_predictions = threshold_predictions.query("model == 'selected'").drop(columns="model").rename(
    columns={"prediction": "predicted_purchase_probability"})
validation_predictions.to_csv(TABLES / "logistic_development_predictions.csv", index=False)
print("Frozen development thresholds:", {k: threshold_metadata[k] for k in ["original", "selected"]})


Frozen development thresholds: {'original': 0.12903448374883927, 'selected': 0.15877993769334672}


## Retrospective CP07 comparison
The fair comparison refits both the original raw-feature logistic model and the selected model on CP01-CP06. The previously published model used only CP01-CP05 for its CP07 evaluation; that result is reproduced separately so gains from extra training data are not confused with gains from feature engineering. No configuration is changed in response to CP07.

In [4]:
original_published = make_propensity_pipe(NUM, CAT, C=.01).fit(
    df.loc[df.campaign_id.lt("CP06"), X_COLS], df.loc[df.campaign_id.lt("CP06"), "purchase_14d"])
original_refit = make_propensity_pipe(NUM, CAT, C=.01).fit(development[X_COLS], development.purchase_14d)
model = fit_purchase_model(development, NUM, CAT, CONFIG)
yte = test.purchase_14d.to_numpy()
p_published = original_published.predict_proba(test[X_COLS])[:, 1]
p_original = original_refit.predict_proba(test[X_COLS])[:, 1]
p_te = model.predict_proba(test[X_COLS])[:, 1]
comparison = pd.DataFrame([
    {"model": "original_published_reproduced", "training_campaigns": "CP01-CP05", **purchase_metrics(yte, p_published)},
    {"model": "original_fair_refit", "training_campaigns": "CP01-CP06", **purchase_metrics(yte, p_original, threshold=ORIGINAL_THR)},
    {"model": "selected", "training_campaigns": "CP01-CP06", **purchase_metrics(yte, p_te, threshold=F1_THR)}])
comparison.to_csv(TABLES / "logistic_cp07_comparison.csv", index=False)
vp = validation_predictions.query("campaign_id == 'CP06'")
res = pd.DataFrame([
    {"split": "valid_CP06", **metrics(vp.purchase_14d, vp.predicted_purchase_probability)},
    {"split": "test_CP07", **metrics(yte, p_te)},
    {"split": "valid_CP06_f1thr", **metrics(vp.purchase_14d, vp.predicted_purchase_probability, F1_THR)},
    {"split": "test_CP07_f1thr", **metrics(yte, p_te, F1_THR)}])
res.insert(1, "C", CONFIG["C"]); res.insert(2, "feature_mode", CONFIG["mode"])
res.to_csv(TABLES / "logistic_metrics.csv", index=False)
headline = comparison.query("model in ['original_fair_refit', 'selected']").copy()
headline.insert(1, "split", "CP07")
headline["threshold_source"] = "CP04-CP06 out-of-time max F1"
headline.to_csv(TABLES / "purchase_headline_metrics.csv", index=False)
headline.assign(model=headline.model.replace({"original_fair_refit": "original"})).to_csv(
    EXPERIMENTS / "cp07_comparison.csv", index=False)
headline[["model", "average_precision", "recall", "threshold", "precision", "predicted_positive_share", "tp", "fn", "fp"]].round(6)


,model,average_precision,recall,threshold,precision,predicted_positive_share,tp,fn,fp
1,original_fair_refit,0.232325,0.608782,0.129034,0.201987,0.397368,305.0,196.0,1205.0
2,selected,0.236708,0.500998,0.158780,0.214897,0.307368,251.0,250.0,917.0


## Uncertainty in the headline metrics
Paired customer-bootstrap draws retain both models' predictions. Average precision and recall changes are evaluated on the same sampled customers; the two development-selected thresholds are held fixed in every draw. Supporting probability-quality intervals remain available. Intervals condition on the fitted models and thresholds, omit model/threshold selection and future-campaign uncertainty, and are retrospective because CP07 was previously inspected.


In [5]:
paired = pd.concat([
    test[["assignment_id", "customer_id", "campaign_id", "purchase_14d"]].assign(model="original", prediction=p_original),
    test[["assignment_id", "customer_id", "campaign_id", "purchase_14d"]].assign(model="selected", prediction=p_te)], ignore_index=True)
paired.to_csv(TABLES / "logistic_paired_test_predictions.csv", index=False)
intervals = paired_purchase_intervals(paired, n_bootstrap=2000, seed=42, thresholds=threshold_metadata)
intervals.to_csv(TABLES / "logistic_cp07_paired_bootstrap.csv", index=False)
paired.to_csv(EXPERIMENTS / "cp07_predictions.csv", index=False)
intervals.to_csv(EXPERIMENTS / "cp07_paired_bootstrap.csv", index=False)
intervals.query("metric in ['average_precision', 'recall']").round(6)


,metric,original,selected,difference_selected_minus_original,ci_low,ci_high,better_direction,bootstrap_draws,bootstrap_unit,original_threshold,selected_threshold,thresholds_fixed_in_bootstrap
0,average_precision,0.232325,0.236708,0.004384,-0.004445,0.013209,higher,2000,customer,NaN,NaN,True
1,recall,0.608782,0.500998,-0.107784,-0.135243,-0.082807,higher,2000,customer,0.129034,0.15878,True


## Ranking and calibration diagnostics
The plots compare original and selected probabilities on the same CP07 population. Decile 1 contains the highest predicted purchase probabilities. These predictions do not identify which offer changes a customer's behavior.

In [6]:
te = test[["assignment_id", "customer_id", "campaign_id", "purchase_14d"]].assign(predicted_purchase_probability=p_te)
te.to_csv(TABLES / "logistic_test_predictions.csv", index=False)
te["decile"] = pd.qcut(te.predicted_purchase_probability.rank(method="first"), 10, labels=range(10, 0, -1))
lift = te.groupby("decile", observed=True).agg(n=("purchase_14d", "size"), purchase_rate=("purchase_14d", "mean"),
                                             mean_pred=("predicted_purchase_probability", "mean")).sort_index()
lift["lift"] = lift.purchase_rate / te.purchase_14d.mean()
lift.to_csv(TABLES / "logistic_test_decile_lift.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
limit = .4
ax[0].plot([0, limit], [0, limit], color="#898781", lw=1, ls="--", label="perfect")
calibration_rows = []
for label, probability, color in [("original refit", p_original, "#898781"), ("selected", p_te, "#2a78d6")]:
    observed, predicted = calibration_curve(yte, probability, n_bins=10, strategy="quantile")
    ax[0].plot(predicted, observed, marker="o", ms=4, lw=1.5, color=color, label=label)
    frame = pd.DataFrame({"prediction": probability, "purchase": yte})
    frame["band"] = pd.qcut(frame.prediction, q=min(10, frame.prediction.nunique()), labels=False, duplicates="drop")
    bins = frame.groupby("band", observed=True).agg(n=("purchase", "size"),
        mean_predicted_probability=("prediction", "mean"), observed_purchase_rate=("purchase", "mean")).reset_index()
    calibration_rows.append(bins.assign(model=label, split="CP07", binning="within-model prediction quantiles; descriptive"))
pd.concat(calibration_rows, ignore_index=True).to_csv(TABLES / "purchase_probability_calibration.csv", index=False)
ax[0].set_xlabel("Mean predicted probability"); ax[0].set_ylabel("Observed purchase rate")
ax[0].legend(frameon=False); ax[0].set_title("CP07 calibration, quantile bins", loc="left", fontsize=10)
ax[1].bar(lift.index.astype(str), lift.purchase_rate, color="#2a78d6", width=.6)
ax[1].axhline(te.purchase_14d.mean(), color="#0b0b0b", lw=.8, ls="--")
ax[1].set_xlabel("Predicted purchase decile (1 = highest)")
ax[1].set_title("Observed purchase rate by decile", loc="left", fontsize=10)
for a in ax:
    a.spines[["top", "right"]].set_visible(False); a.grid(alpha=.2)
fig.tight_layout(); fig.savefig(FIGS / "05_logistic_calibration_lift.png", bbox_inches="tight"); plt.show()
lift.round(3)


C:\Users\yifan\AppData\Local\Temp\ipykernel_34052\866159469.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS / "05_logistic_calibration_lift.png", bbox_inches="tight"); plt.show()


,n,purchase_rate,mean_pred,lift
decile,,,,
10,380,0.055,0.060,0.419
9,380,0.061,0.066,0.459
8,380,0.095,0.073,0.719
7,380,0.100,0.082,0.758
6,380,0.087,0.091,0.659
5,380,0.116,0.114,0.878
4,380,0.158,0.147,1.198
3,380,0.166,0.171,1.257
2,380,0.171,0.206,1.297


## Model interpretation and July scoring artifact
Coefficients are conditional associations on transformed, standardized, and one-hot features; correlated predictors and interactions prevent interpreting them as causal effects. After evaluation, refit the frozen configuration on CP01-CP07 for July scoring. The refit uses CP07 labels only after the comparison and is not used to evaluate CP07.

In [7]:
base = getattr(model, "base", model)
coef = pd.Series(base.named_steps["lr"].coef_[0], index=base.named_steps["pre"].get_feature_names_out()).sort_values()
coef.rename("coefficient").to_csv(TABLES / "logistic_coefficients.csv")
display(pd.concat([coef.head(8), coef.tail(8)]).round(3).to_frame("coefficient"))
final = fit_purchase_model(df, NUM, CAT, CONFIG)
joblib.dump(final, MODELS / "logistic_propensity_model.joblib")
final_base = getattr(final, "base", final)
spec = {"target": "purchase_14d", "C": CONFIG["C"], "configuration": CONFIG,
        "numeric_features": NUM, "categorical_features": CAT,
        "encoded_features": final_base.named_steps["pre"].get_feature_names_out().tolist(),
        "train_campaigns_final": sorted(df.campaign_id.unique()),
        "validation_campaigns": ["CP04", "CP05", "CP06"],
        "evaluation_fit_campaigns": sorted(development.campaign_id.unique()),
        "evaluation_campaigns": ["CP07"], "evaluation_status": "retrospective; CP07 was previously inspected",
        "selection_rule": PURCHASE_SELECTION_RULE, "f1_threshold_from_valid": F1_THR,
        "headline_metrics": ["average_precision", "recall"], "threshold_selection": threshold_metadata,
        "probability_interpretation": "purchase under historical treatment mixture, not untreated purchase probability"}
(MODELS / "logistic_model_features.json").write_text(json.dumps(spec, indent=2), encoding="utf-8")
reloaded = joblib.load(MODELS / "logistic_propensity_model.joblib")
np.testing.assert_allclose(reloaded.predict_proba(test[X_COLS]), final.predict_proba(test[X_COLS]))
assert np.isfinite(res[["average_precision", "recall", "roc_auc", "log_loss", "brier"]].values).all()
print(f"Saved + reloaded: {len(X_COLS)} raw predictors -> {len(spec['encoded_features'])} encoded features")


,coefficient
cat__customer_tier_Standard,-0.432
cat__tier_membership_Standard_0,-0.236
cat__tier_membership_Standard_1,-0.196
num__preferred_matches_browse,-0.133
num__missingindicator_days_since_last_email_click,-0.086
num__missingindicator_session_recency_intensity,-0.075
num__missingindicator_days_since_last_session,-0.075
cat__preferred_category_Accessories,-0.072
cat__customer_tier_Silver,0.057
cat__top_category_viewed_30d_Apparel,0.058


Saved + reloaded: 43 raw predictors -> 103 encoded features
